# 06 - Fuzzy Matching

## Objective

Menguji fuzzy similarity hanya pada candidate pair hasil deterministic blocking. Tidak dilakukan perbandingan seluruh dataset.

## Research Questions

1. Bagaimana distribusi skor similarity pada candidate pair deterministic?
2. Bagaimana jumlah kandidat berubah pada threshold 80, 90, dan 95?
3. Apakah skor multi-field lebih informatif daripada satu field saja?
4. Apakah threshold tertentu dapat dipilih tanpa ground truth?

## Hypothesis

- Candidate pair yang didukung beberapa exact rule cenderung memiliki similarity nama dan alamat lebih tinggi.
- Threshold tinggi akan mengurangi kandidat, tetapi dapat menurunkan recall.
- Tanpa ground truth, threshold hanya dapat dibandingkan sebagai sensitivity analysis, bukan dipilih sebagai threshold final.

## Scope and limitations

- Fuzzy scoring hanya diterapkan pada candidate pair hasil blocking R1-R4.
- Tidak ada N x N comparison.
- Tidak ada automatic merge atau penghapusan baris.
- `customer_id` hanya dipakai untuk audit internal, bukan sebagai label ground truth.
- Nilai customer tidak ditampilkan; output disimpan sebagai row index dan agregat skor.

In [1]:
from itertools import combinations
from pathlib import Path
from time import perf_counter
import pandas as pd
from rapidfuzz import fuzz

DATA_CANDIDATES = [
    Path.cwd() / 'data' / 'processed' / 'crm_50000_customers_standardized.csv',
    Path.cwd().parent / 'data' / 'processed' / 'crm_50000_customers_standardized.csv',
]
DATA_PATH = next((path.resolve() for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Dataset terstandardisasi tidak ditemukan.')

df = pd.read_csv(DATA_PATH).reset_index(names='row_index')
print('File:', DATA_PATH)
print('Shape:', df.shape)

File: C:\Users\User\Documents\Maganghub 2026\Bulan 1\Tes duplikasi\data\processed\crm_50000_customers_standardized.csv
Shape: (50000, 25)


## Experiment 1 - Rebuild deterministic candidate pairs

Candidate pair direkonstruksi dari aturan R1-R4 agar notebook ini reproducible dan tidak bergantung pada state notebook 05.

In [2]:
name_dob_parts = df[['name_key_std', 'dob_std']]
df['name_dob_key'] = name_dob_parts.fillna('').astype('string').agg('|'.join, axis=1)
df.loc[name_dob_parts.isna().any(axis=1), 'name_dob_key'] = pd.NA
email_phone_parts = df[['email_std', 'phone_digits_std']]
df['email_phone_key'] = email_phone_parts.fillna('').astype('string').agg('|'.join, axis=1)
df.loc[email_phone_parts.isna().any(axis=1), 'email_phone_key'] = pd.NA

pair_rules = {}
rule_definitions = [
    ('email_std', 'R1_exact_email'),
    ('phone_digits_std', 'R2_exact_phone'),
    ('name_dob_key', 'R3_name_plus_dob'),
    ('email_phone_key', 'R4_email_plus_phone'),
]
for key, rule in rule_definitions:
    for _, block in df.dropna(subset=[key]).groupby(key, sort=False):
        row_indices = sorted(block['row_index'].tolist())
        for left_index, right_index in combinations(row_indices, 2):
            pair_rules.setdefault((left_index, right_index), set()).add(rule)

candidate_pairs = pd.DataFrame([
    {
        'left_row_index': left_index,
        'right_row_index': right_index,
        'supporting_rules': '|'.join(sorted(rules)),
        'rule_count': len(rules),
    }
    for (left_index, right_index), rules in sorted(pair_rules.items())
])
print('Candidate pairs:', len(candidate_pairs))

Candidate pairs: 7106


## Experiment 2 - Multi-field fuzzy score

Skor menggunakan `fuzz.ratio` pada nama, alamat, dan kota. Nilai akhir adalah rata-rata hanya dari field yang tersedia pada kedua row.

Threshold 80, 90, dan 95 dipakai sebagai sensitivity analysis. Angka tersebut bukan threshold final karena ground truth belum tersedia.

In [3]:
def text_value(value):
    if pd.isna(value):
        return None
    return str(value)

def pair_score(left_row, right_row):
    scores = {}
    for field in ['name_key_std', 'address_std', 'city']:
        left_value = text_value(left_row[field])
        right_value = text_value(right_row[field])
        if left_value is not None and right_value is not None:
            scores[field] = fuzz.ratio(left_value, right_value)
    if not scores:
        return pd.Series({'name_score': pd.NA, 'address_score': pd.NA, 'city_score': pd.NA, 'mean_score': pd.NA})
    return pd.Series({
        'name_score': scores.get('name_key_std', pd.NA),
        'address_score': scores.get('address_std', pd.NA),
        'city_score': scores.get('city', pd.NA),
        'mean_score': sum(scores.values()) / len(scores),
    })

start_time = perf_counter()
score_rows = []
for pair in candidate_pairs.itertuples(index=False):
    left_row = df.iloc[pair.left_row_index]
    right_row = df.iloc[pair.right_row_index]
    scores = pair_score(left_row, right_row)
    score_rows.append({
        'left_row_index': pair.left_row_index,
        'right_row_index': pair.right_row_index,
        'supporting_rules': pair.supporting_rules,
        'rule_count': pair.rule_count,
        **scores.to_dict(),
    })
fuzzy_scores = pd.DataFrame(score_rows)
runtime_seconds = perf_counter() - start_time
print('Scored pairs:', len(fuzzy_scores))
print(f'Runtime seconds: {runtime_seconds:.3f}')
fuzzy_scores[['name_score', 'address_score', 'city_score', 'mean_score']].describe().round(2)

Scored pairs: 7106
Runtime seconds: 1.804


,name_score,address_score,city_score,mean_score
count,7106.00,7106.00,7106.00,7106.00
mean,47.66,51.66,47.14,48.82
std,31.39,29.80,33.13,30.61
min,0.00,10.53,0.00,13.90
25%,25.00,30.19,23.08,28.19
50%,33.33,36.84,31.58,32.91
75%,88.00,100.00,100.00,96.00
max,100.00,100.00,100.00,100.00


In [4]:
thresholds = [80, 90, 95]
threshold_rows = []
for threshold in thresholds:
    selected = fuzzy_scores[fuzzy_scores['mean_score'] >= threshold]
    threshold_rows.append({
        'threshold': threshold,
        'candidate_pairs_above_threshold': len(selected),
        'percentage_of_candidate_pairs': len(selected) / len(fuzzy_scores) * 100,
        'pairs_with_2_or_more_rules': int((selected['rule_count'] >= 2).sum()),
    })
threshold_summary = pd.DataFrame(threshold_rows)
threshold_summary.round(2)

,threshold,candidate_pairs_above_threshold,percentage_of_candidate_pairs,pairs_with_2_or_more_rules
0,80,1867,26.27,1861
1,90,1867,26.27,1861
2,95,1833,25.80,1827


In [5]:
rule_score_summary = (
    fuzzy_scores.groupby('rule_count')
    .agg(pair_count=('mean_score', 'size'), mean_score=('mean_score', 'mean'), median_score=('mean_score', 'median'))
    .reset_index()
)
rule_score_summary.round(2)

,rule_count,pair_count,mean_score,median_score
0,1,5245,30.88,30.24
1,2,35,100.00,100.00
2,3,380,97.02,96.77
3,4,1446,100.00,100.00


## Experiment 3 - Internal audit by threshold

Audit ini membandingkan hasil threshold dengan `customer_id` hanya sebagai consistency check. Ini bukan evaluasi ground truth.

In [6]:
audit_base = fuzzy_scores.merge(df[['row_index', 'customer_id']], left_on='left_row_index', right_on='row_index', how='left').rename(columns={'customer_id': 'left_customer_id'}).drop(columns='row_index')
audit_base = audit_base.merge(df[['row_index', 'customer_id']], left_on='right_row_index', right_on='row_index', how='left').rename(columns={'customer_id': 'right_customer_id'}).drop(columns='row_index')
audit_base['same_customer_id'] = audit_base['left_customer_id'] == audit_base['right_customer_id']
audit_rows = []
for threshold in thresholds:
    selected = audit_base[audit_base['mean_score'] >= threshold]
    audit_rows.append({
        'threshold': threshold,
        'selected_pairs': len(selected),
        'same_customer_id': int(selected['same_customer_id'].sum()),
        'different_customer_id': int((~selected['same_customer_id']).sum()),
        'same_customer_id_percentage': selected['same_customer_id'].mean() * 100 if len(selected) else 0,
    })
threshold_audit = pd.DataFrame(audit_rows)
threshold_audit.round(2)

,threshold,selected_pairs,same_customer_id,different_customer_id,same_customer_id_percentage
0,80,1867,1867,0,100.0
1,90,1867,1867,0,100.0
2,95,1833,1833,0,100.0


In [7]:
OUTPUT_DIR = DATA_PATH.parents[1] / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCORE_OUTPUT_PATH = OUTPUT_DIR / 'deterministic_candidate_fuzzy_scores.csv'
fuzzy_scores.to_csv(SCORE_OUTPUT_PATH, index=False)
print('Saved score table:', SCORE_OUTPUT_PATH)
print('Raw dataset still exists:', (DATA_PATH.parents[1] / 'raw' / 'crm_50000_customers_dirty_v3.csv').exists())

Saved score table: C:\Users\User\Documents\Maganghub 2026\Bulan 1\Tes duplikasi\data\processed\deterministic_candidate_fuzzy_scores.csv
Raw dataset still exists: True


# Result, Analysis, and Decision

## Result aktual

- Candidate pair deterministic yang direkonstruksi: `7.106` pair.
- Fuzzy scoring selesai pada `7.106` pair dalam runtime `1,804` detik pada environment saat eksperimen dijalankan.
- Mean score: median `32,91`, kuartil ketiga `96,00`, maksimum `100,00`.
- Threshold `80`: `1.867` pair atau `26,27%` dari candidate pair.
- Threshold `90`: `1.867` pair atau `26,27%`.
- Threshold `95`: `1.833` pair atau `25,80%`.
- Pada threshold 80, 90, dan 95, seluruh pair terpilih memiliki `customer_id` sama dalam audit internal.
- Pair dengan 2 supporting rules memiliki mean/median score `100,00`; pair dengan 3 rules memiliki mean `97,02` dan median `96,77`; pair dengan 1 rule memiliki mean `30,88` dan median `30,24`.

## Analysis

- Skor fuzzy terutama memisahkan candidate pair yang hanya didukung satu exact rule dari pair yang didukung beberapa rule.
- Threshold 80 dan 90 belum membedakan hasil karena distribusi skor memiliki gap pada candidate pair terpilih.
- Threshold 95 mengurangi 34 pair, tetapi tidak mengubah audit internal pada snapshot ini.
- Hasil `customer_id` yang sama adalah consistency check terhadap identifier dataset, bukan ground truth eksternal. Tidak boleh dilaporkan sebagai precision atau recall.

## Decision

Fuzzy matching tidak perlu dijalankan pada seluruh dataset. Untuk eksperimen lanjutan, kandidat prioritas adalah pair dengan minimal dua supporting rules atau mean score minimal 95. Pair yang hanya didukung satu exact email/telepon belum boleh otomatis diterima.

Tidak ada merge, delete, atau perubahan raw dataset. Score table disimpan terpisah.

## Next Experiment

Lanjutkan ke `07_blocking_candidate_generation.ipynb` hanya jika ingin menguji efisiensi blocking secara terpisah. Jika fokusnya evaluasi kualitas kandidat, tahap berikutnya lebih tepat `09_evaluation.ipynb`, dengan catatan ground truth belum tersedia.